## Bronze Layer — Gear Catalogue
Fetches all gear items (shoes, bikes) and their usage stats from the Garmin API.
Full refresh on every run — gear catalogue is small and changes infrequently.
Two tables: raw.gear_list and raw.gear_stats

In [0]:
%pip install garminconnect

In [0]:
import pandas as pd
from garminconnect import Garmin
import json

In [0]:
email = dbutils.secrets.get(scope="garmin", key="email")
password = dbutils.secrets.get(scope="garmin", key="password")
print("Credentials loaded ✅")

In [0]:
api = Garmin(email=email, password=password)
api.login()
print("Garmin API connected ✅")

In [0]:
device_info = api.get_device_last_used()
user_profile_number = device_info.get("userProfileNumber")
print(f"User profile number: {user_profile_number}")

In [0]:
# Fetch gear list
gear_list = api.get_gear(user_profile_number)
df_gear = pd.DataFrame(gear_list)

print(f"Found {len(df_gear)} gear items")
print(f"Columns: {list(df_gear.columns)}")
df_gear.display()

In [0]:
# Fetch gear stats for each item
gear_stats_list = []

for gear_item in gear_list:
    gear_uuid = gear_item['uuid']
    try:
        stats = api.get_gear_stats(gear_uuid)
        gear_stats_list.append(stats)
        print(f"Fetched stats for {gear_item.get('customMakeModel', gear_uuid)} ✅")
    except Exception as e:
        print(f"Warning: Could not get stats for {gear_uuid}: {e}")

df_gear_stats = pd.DataFrame(gear_stats_list)
print(f"\nGear stats rows: {len(df_gear_stats)}")
print(f"Columns: {list(df_gear_stats.columns)}")

In [0]:
# Drop image columns from gear_list to avoid void types
image_cols = ['imageNameLarge', 'imageNameMedium', 'imageNameSmall']
df_gear = df_gear.drop(columns=[col for col in image_cols if col in df_gear.columns])

# Serialise complex fields in gear_list
for col_name in df_gear.columns:
    if df_gear[col_name].apply(lambda x: isinstance(x, (dict, list))).any():
        df_gear[col_name] = df_gear[col_name].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )

print(f"Gear List - Columns: {len(df_gear.columns)}, Rows: {len(df_gear)}")
print(f"Gear Stats - Columns: {len(df_gear_stats.columns)}, Rows: {len(df_gear_stats)}")

# Write gear_list
spark.createDataFrame(df_gear) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("garmin_lakehouse.raw.gear_list")

print(f"gear_list written: {spark.table('garmin_lakehouse.raw.gear_list').count()} rows ✅")

# Write gear_stats
spark.createDataFrame(df_gear_stats) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("garmin_lakehouse.raw.gear_stats")

print(f"gear_stats written: {spark.table('garmin_lakehouse.raw.gear_stats').count()} rows ✅")

In [0]:
display(spark.table("garmin_lakehouse.raw.gear_list"))

In [0]:
display(spark.table("garmin_lakehouse.raw.gear_stats"))